In [ ]:
#====================#  1. IMPORT LIBRARIES #====================#
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Reproducibility
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

# Create results folder
os.makedirs("./results", exist_ok=True)

# Plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

print("Libraries loaded successfully.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#

In [ ]:
#====================# 2. LOAD DATA #====================#

train_df = pd.read_csv("/content/drive/MyDrive/NLP DISTILBERT SAMPLE/train.csv")
val_df   = pd.read_csv("/content/drive/MyDrive/NLP DISTILBERT SAMPLE/val.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/NLP DISTILBERT SAMPLE/test.csv")

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

train_df.head()

In [ ]:
#====================#  3. PREPARE FEATURES + LABELS #====================#

X_train = train_df["text"].tolist()
y_train = train_df["label_binary"].tolist()

X_val = val_df["text"].tolist()
y_val = val_df["label_binary"].tolist()

X_test = test_df["text"].tolist()
y_test = test_df["label_binary"].tolist()

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))


In [ ]:
#====================#  4. LOAD TOKENIZER #====================#

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

print("Tokenizer loaded.")


In [ ]:
#====================#  5. TOKENIZATION #====================#

train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    X_val,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization complete.")

In [ ]:
#====================#  6. CREATE DATASET CLASS #====================#

class ScamDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ScamDataset(train_encodings, y_train)
val_dataset   = ScamDataset(val_encodings, y_val)
test_dataset  = ScamDataset(test_encodings, y_test)

print("Datasets created.")

In [ ]:
#====================#  7. LOAD MODEL #====================#

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.config.id2label = {0: "ham", 1: "spam"}
model.config.label2id = {"ham": 0, "spam": 1}


print("Model loaded.")

In [ ]:
#====================#  8. EVALUATION FUNCTION #====================#

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    probs = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()[:, 1]

    accuracy = accuracy_score(labels, preds)

    precision = precision_score(labels, preds)

    recall = recall_score(labels, preds)

    f1 = f1_score(labels, preds)

    roc_auc = roc_auc_score(labels, probs)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

print("Evaluation function ready.")

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_steps=50,

    save_strategy="epoch",

    fp16=torch.cuda.is_available(),

    report_to="none"
)

print("Training arguments configured.")

In [ ]:
#====================# 10. TRAINER #====================#

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

print("Trainer created.")


In [ ]:
#====================# 11. TRAIN MODEL #====================#

trainer.train()

print("Training complete.")

In [ ]:
#====================#  12. VALIDATION EVALUATION #====================#

val_predictions = trainer.predict(val_dataset)

val_preds = np.argmax(
    val_predictions.predictions,
    axis=1
)

val_probs = torch.softmax(
    torch.tensor(val_predictions.predictions),
    dim=1
)[:, 1].numpy()

# Metrics
val_accuracy = accuracy_score(y_val, val_preds)

val_precision = precision_score(y_val, val_preds)

val_recall = recall_score(y_val, val_preds)

val_f1 = f1_score(y_val, val_preds)

val_roc_auc = roc_auc_score(y_val, val_probs)

print("\n=== VALIDATION RESULTS ===")

print(f"Accuracy : {val_accuracy:.4f}")
print(f"Precision: {val_precision:.4f}")
print(f"Recall   : {val_recall:.4f}")
print(f"F1-Score : {val_f1:.4f}")
print(f"ROC-AUC  : {val_roc_auc:.4f}")

print("\n=== VALIDATION CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_val,
        val_preds,
        target_names=["ham", "spam"]
    )
)

In [ ]:
#====================# 13. VALIDATION CONFUSION MATRIX #====================#

fig, ax = plt.subplots(figsize=(5, 4))

cm = confusion_matrix(y_val, val_preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["ham", "spam"]
)

disp.plot(ax=ax, colorbar=False)

ax.set_title("Validation Confusion Matrix")

plt.show()

In [ ]:
#====================# 14. TEST EVALUATION #====================#

test_predictions = trainer.predict(test_dataset)

test_preds = np.argmax(
    test_predictions.predictions,
    axis=1
)

test_probs = torch.softmax(
    torch.tensor(test_predictions.predictions),
    dim=1
)[:, 1].numpy()

# Metrics
test_accuracy = accuracy_score(y_test, test_preds)

test_precision = precision_score(y_test, test_preds)

test_recall = recall_score(y_test, test_preds)

test_f1 = f1_score(y_test, test_preds)

test_roc_auc = roc_auc_score(y_test, test_probs)

print("\n=== TEST RESULTS ===")

print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1-Score : {test_f1:.4f}")
print(f"ROC-AUC  : {test_roc_auc:.4f}")

print("\n=== TEST CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_test,
        test_preds,
        target_names=["ham", "spam"]
    )
)

In [ ]:
#====================# 15. TEST CONFUSION MATRIX #====================#

fig, ax = plt.subplots(figsize=(5, 4))

cm = confusion_matrix(y_test, test_preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["ham", "spam"]
)

disp.plot(ax=ax, colorbar=False)

ax.set_title("Test Confusion Matrix")

plt.show()

In [ ]:
#====================# 16. ERROR ANALYSIS #====================#

results_df = test_df.copy()

results_df["predicted"] = test_preds

results_df["confidence"] = test_probs

# False Positives
fp = results_df[
    (results_df["label_binary"] == 0) &
    (results_df["predicted"] == 1)
]

print("\n=== FALSE POSITIVES ===")
print(fp[["text", "confidence"]].head(10))

# False Negatives
fn = results_df[
    (results_df["label_binary"] == 1) &
    (results_df["predicted"] == 0)
]

print("\n=== FALSE NEGATIVES ===")
print(fn[["text", "confidence"]].head(10))

In [ ]:
#====================# 17. CONFIDENCE SCORE VISUALIZATION #====================#

plt.figure(figsize=(7, 4))

sns.histplot(
    test_probs,
    bins=20,
    kde=True
)

plt.title("Prediction Confidence Distribution")
plt.xlabel("Spam Probability")
plt.ylabel("Count")

plt.show()

In [ ]:
#====================# 18. FINAL SUMMARY #====================#

print("\n==============================")
print("FINAL MODEL SUMMARY")
print("==============================")

print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall   : {test_recall:.4f}")
print(f"Test F1-Score : {test_f1:.4f}")
print(f"Test ROC-AUC  : {test_roc_auc:.4f}")

print("\nDistilBERT evaluation complete.")

In [ ]:
#====================# 19. SAVE METRICS #====================#

metrics_df = pd.DataFrame([
    {
        "split": "validation",
        "accuracy": val_accuracy,
        "precision": val_precision,
        "recall": val_recall,
        "f1": val_f1,
        "roc_auc": val_roc_auc
    },
    {
        "split": "test",
        "accuracy": test_accuracy,
        "precision": test_precision,
        "recall": test_recall,
        "f1": test_f1,
        "roc_auc": test_roc_auc
    }
])

metrics_df.to_csv(
    "./results/distilbert_metrics.csv",
    index=False
)

print("Metrics saved.")

results_df.to_csv(
    "./results/distilbert_predictions.csv",
    index=False
)

print("Predictions saved.")

In [ ]:
#====================# 20. DEMONSTRATION #====================#
import torch
import torch.nn.functional as F

def predict(text):

    # put model in eval mode
    model.eval()

    # tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # move to GPU if available
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # disable gradients
    with torch.no_grad():
        outputs = model(**inputs)

    # softmax probabilities
    probs = F.softmax(outputs.logits, dim=1)[0]

    # get prediction
    pred_class = torch.argmax(probs).item()

    label_map = {0: "ham", 1: "spam"}

    return {
        "text": text,
        "prediction": label_map[pred_class],
        "confidence": float(probs[pred_class])
    }

In [ ]:
trainer.evaluate(train_dataset)

print(predict("how're you doing, free this evening"))
print(predict("CLICK THIS TO GET A FREE XBOX!!"))